# Quick checks before training (no training, ~20-30 min on one GPU)

**Check 1, pure noise (H3):** add random noise to the weights of a checkpoint (no retraining) and measure self vs task change. If the self still "changes less" under pure noise, part of our gap comes from how we measure, not from learning. Also gives the noise-size -> movement curve used to size the Q2 push.

**Check 2, far-apart checkpoints:** Hindi only, phase p vs p+k. Does natural drift reach the continual level of movement (~45-50% task change), and is the self still protected there? Plus continual same-language pairs and ordinary switches as the reference.

Fixed tau = 0.2, MLP layers, FLORES dev inputs, phases <= 60. Outputs: `analysis/v6/quick_checks/`.

In [ ]:
import os, sys, glob
REPO = "/workspace/shreshth"          # <- your repo folder on the instance
os.chdir(REPO); print(os.getcwd())
GPU = "0"

In [ ]:
# 1. checkpoints this script needs (downloads only these from your HF model repo if missing)
#    files are named p{phase:03d}_{lang}.pt
from huggingface_hub import HfApi, hf_hub_download
u = HfApi().whoami()["name"]
lang_c = lambda ph: ["hi", "zh", "ar"][(ph - 1) % 3]          # continual order hi -> zh -> ar
need = {"v4b_continual_s0": [(p, lang_c(p)) for p in [30, 33, 44, 45, 46, 47, 59, 60]],
        "v4b_only_hi_s0":   [(p, "hi") for p in [2, 10, 30, 31, 35, 40, 45, 50, 60]]}
for run, items in need.items():
    for ph, l in items:
        rel = f"{run}/ckpt/p{ph:03d}_{l}.pt"
        if not os.path.exists(f"runs/{rel}"):
            print("downloading", rel)
            hf_hub_download(f"{u}/cml-lang-self-checkpoints", rel, repo_type="model", local_dir="runs")
print("checkpoints ok:", all(os.path.exists(f"runs/{r}/ckpt/p{p:03d}_{l}.pt") for r, it in need.items() for p, l in it))

In [ ]:
# 2. FLORES dev files (en, hi, zh, ar). Skips if already there.
import json
codes = {"en": "eng_Latn", "hi": "hin_Deva", "zh": "cmn_Hans", "ar": "arb_Arab"}
os.makedirs("data/flores", exist_ok=True)
missing = [l for l in codes if not os.path.exists(f"data/flores/dev.{l}.jsonl")]
if missing:
    from datasets import load_dataset          # gated dataset: accept the terms on HF once, and be logged in
    for l in missing:
        ds = load_dataset("openlanguagedata/flores_plus", codes[l], split="dev")
        with open(f"data/flores/dev.{l}.jsonl", "w", encoding="utf-8") as f:
            for t in ds["text"]:
                f.write(json.dumps({"text": t}, ensure_ascii=False) + "\n")
        print("wrote", l, len(ds))
print({l: sum(1 for _ in open(f"data/flores/dev.{l}.jsonl")) for l in codes})   # should be 997 each

In [ ]:
# 3. run (prints a line per comparison as it goes; ~20-30 min)
!python -u quick_checks.py --gpu {GPU}

In [ ]:
# 4. results
from IPython.display import Image, display
print(open("analysis/v6/quick_checks/summary.txt").read())
display(Image("analysis/v6/quick_checks/q2_movement_vs_gap.png"))